# Parallel Qoruyo OCR on Kaggle

Run one copy of this notebook on each Kaggle machine. Set the same `N_PARTS` everywhere and give each machine a different `PART_INDEX` from 1 through `N_PARTS`. Each run processes a deterministic, non-overlapping slice of the numerically ordered images and creates a ZIP archive of its OCR output.

Replace the placeholder paths in the configuration cell with paths from your own Kaggle input dataset. Select the Qoruyo recognition model that matches the printed script: Serto, Estrangela, or Eastern Syriac.

In [ ]:
%pip install -q kraken

In [ ]:
from pathlib import Path

PART_INDEX = 1  # Give each machine a unique value: 1, 2, ..., N_PARTS
N_PARTS = 4

IMAGE_DIR = Path("/kaggle/input/YOUR_DATASET/YOUR_IMAGE_FOLDER")
SEG_MODEL = Path("/kaggle/input/YOUR_DATASET/YOUR_MODEL_FOLDER/syr_print_seg03_91.mlmodel")
OCR_MODEL = Path("/kaggle/input/YOUR_DATASET/YOUR_MODEL_FOLDER/YOUR_RECOGNITION_MODEL.mlmodel")

OUTPUT_DIR = Path(f"/kaggle/working/qoruyo_ocr_part_{PART_INDEX}")
ZIP_PATH = Path(f"/kaggle/working/qoruyo_ocr_part_{PART_INDEX}.zip")
BATCH_SIZE = 50
RECURSIVE = True
OVERWRITE = False

In [ ]:
import math
import os
import re
import subprocess
import zipfile

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

def natural_key(path):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", path.name)]

def batches(items, size):
    for index in range(0, len(items), size):
        yield items[index:index + size]

for path, label in ((IMAGE_DIR, "image directory"), (SEG_MODEL, "segmentation model"), (OCR_MODEL, "OCR model")):
    assert path.exists(), f"Missing {label}: {path}"
assert 1 <= PART_INDEX <= N_PARTS, "PART_INDEX must be between 1 and N_PARTS"
assert BATCH_SIZE >= 1, "BATCH_SIZE must be at least 1"

candidates = IMAGE_DIR.rglob("*") if RECURSIVE else IMAGE_DIR.glob("*")
images = sorted(
    (path for path in candidates if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES),
    key=natural_key,
)
assert images, f"No supported images found in {IMAGE_DIR}"

duplicate_stems = {}
for path in images:
    duplicate_stems.setdefault(path.stem, []).append(path)
conflicts = [stem for stem, paths in duplicate_stems.items() if len(paths) > 1]
assert not conflicts, f"Duplicate image stems would overwrite OCR output: {conflicts}"

part_size = math.ceil(len(images) / N_PARTS)
start = (PART_INDEX - 1) * part_size
end = min(PART_INDEX * part_size, len(images))
selected = images[start:end]

print(f"Found {len(images)} total image(s)")
print(f"Part {PART_INDEX}/{N_PARTS}: indexes {start}:{end}, {len(selected)} image(s)")
if selected:
    print(f"First: {selected[0].name}")
    print(f"Last:  {selected[-1].name}")

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
failures = []

for batch_number, image_batch in enumerate(batches(selected, BATCH_SIZE), start=1):
    command = ["kraken"]
    queued = []

    for image_path in image_batch:
        output_path = OUTPUT_DIR / f"{image_path.stem}.txt"
        if not OVERWRITE and output_path.exists() and output_path.stat().st_size > 0:
            continue
        command.extend(["-i", str(image_path), str(output_path)])
        queued.append(image_path)

    if not queued:
        print(f"Batch {batch_number}: already complete")
        continue

    command.extend([
        "segment", "-bl", "-i", str(SEG_MODEL), "-d", "horizontal-rl",
        "ocr", "-m", str(OCR_MODEL), "--reorder", "--base-dir", "R",
    ])

    print(f"Batch {batch_number}: {queued[0].name} through {queued[-1].name}")
    result = subprocess.run(command, env=environment, text=True, capture_output=True)
    if result.returncode != 0:
        failures.append((batch_number, len(queued), result.returncode))
        print(result.stdout)
        print(result.stderr)
    else:
        print(f"Batch {batch_number}: complete")

print(f"Failed batches: {failures}")

In [ ]:
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for text_path in sorted(OUTPUT_DIR.glob("*.txt"), key=natural_key):
        archive.write(text_path, arcname=text_path.name)

print(f"OCR files in this part: {len(list(OUTPUT_DIR.glob('*.txt')))}")
print(f"Created: {ZIP_PATH}")